# ATP Value Betting: Logistic Regression Three Ways

## Decisions

* **Business scenario:** The idea is a betting syndicate that wants to make as much profit as possible from a portfolio of single bets on ATP matches. The model doesn't have to beat the bookmaker on every match, that would be unrealistic. It just needs to find the matches where the bookamker's price looks off, and bet only on those.

* **Target:** I reframed every match as Player A vs Player B, with the two players assigned to each side randomly, and y = 1 if Player A won. I did this becuase in the raw Winner/Loser format the column a value sits in basically tells you who won, so the model would be cheating.

* **Split:** I split by time, not randomly. The syndicate always uses past matches to predict future ones, so with a random split the model would be learning from matches played after the ones its being tested on. 2018 is only a warm-up season, since player histories start from zero and the features for that year aren't reliable. Then 2019-2023 is training, 2024 is validation (wich I use for tuning) and 2025 is the test season that I don't touch until the very end.

* **Data cleaning:** I only kept completed matches. Walkovers were never actually played, and retirements depend on an injury more than on the tennis itself, so keeping them would just add noise to both the labels and the player histories.

* **Odds:** I use the Bet365 odds as the benchmark to beat and to calculate profit, but not as a feature. If I used them as a feature the model would just end up copying the market and all my feature engineering would be pretty much useless. The question I actually want to answer is wether public information on its own is enough to find value against the market.

* **Decision metric:** I look at profit and ROI across the portfolio of bets, not accuracy. A model can pick the winner most of the times and still lose money if it keeps betting at bad prices. A bet is placed only when the model's expected value (probability x odds - 1) is above a minimum edge, and I choose that threshold on the validation set.

## 0. Data source and acquisition (one-off, not rerun)
Source: tennis-data.co.uk, ATP main-tour results, one Excel file per season.
Seasons 2018-2025 downloaded manually (the site blocks scripted downloads).
2026 excluded because the season is incomplete.
The 8 files were combined once into `data/atp_matches_2018_2025.csv` (19,685 matches, 43 columns),
which is the file committed to the repo and loaded below:

```python
SEASONS = range(2018, 2026)
frames = []
for year in SEASONS:
    df_y = pd.read_excel(f"data/{year}.xlsx")
    df_y["season"] = year
    frames.append(df_y)
raw = pd.concat(frames, ignore_index=True)
raw.to_csv("data/atp_matches_2018_2025.csv", index=False)
```

## 1. Setup and data loading

In [18]:
import numpy as np
import pandas as pd
from collections import defaultdict, deque
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

df = pd.read_csv("data/atp_matches_2018_2025.csv")
df["Date"] = pd.to_datetime(df["Date"])
print(df.shape)

(19685, 43)


## 2. Cleaning and Player A / Player B reshape
Raw rows are written from the winner's side (Winner/Loser columns), which leaks the outcome.
Each match is randomly assigned to A/B; target y = 1 if A won.

In [19]:
# Decision: keep only completed matches (drop retirements, walkovers, etc.)
df = df[df["Comment"] == "Completed"].copy()

# Chronological order matters for everything that follows
df = df.sort_values("Date", kind="stable").reset_index(drop=True)

# Randomly decide, per match, whether the winner becomes A or B
rng = np.random.default_rng(42)
swap = rng.random(len(df)) < 0.5

def assign(winner_col, loser_col):
    a = np.where(swap, df[loser_col], df[winner_col])
    b = np.where(swap, df[winner_col], df[loser_col])
    return a, b

df["A"], df["B"] = assign("Winner", "Loser")
df["A_rank"], df["B_rank"] = assign("WRank", "LRank")
df["A_odds"], df["B_odds"] = assign("B365W", "B365L")
df["y"] = (~swap).astype(int)   # 1 if A won

print(df.shape, df["y"].mean())

(19017, 50) 0.5019719198611768


19,685 → 19,017 matches after keeping only completed matches; y mean is 0.502, so classes are balanced.

**Why retirements and walkovers are removed everywhere, player histories included:** a walkover is a match that was never played, and a retirement is decided by an injury rather than by which player was better. Keeping them would bias the Elo ratings and form features, not only the evaluation. The 93 matches with missing or invalid odds (section 3) are treated differently: they were completed normally and have a valid result, so they still update player histories and are dropped only from the evaluation, where odds are required. One limitation is that a bettor cannot know beforehand that a match will end in a retirement, so removing these matches from the test season makes the results look slightly better than they would be in practice.

## 3. Feature engineering (pre-match information only)
For each match in date order: (1) build features from each player's history, (2) only then update the history with the result.

In [20]:
K = 32                                   # Elo update speed
def expected(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

elo = defaultdict(lambda: 1500.0)
elo_surf = defaultdict(lambda: 1500.0)
recent = defaultdict(lambda: deque(maxlen=10))
surf_rec = defaultdict(lambda: [0, 0])
dates = defaultdict(list)

rows = []
for r in df.itertuples(index=False):
    a, b, s, d, y = r.A, r.B, r.Surface, r.Date, r.y

    # 1) FEATURES: only past information
    def form(p):
        return np.mean(recent[p]) if recent[p] else 0.5
    def surf_wr(p):
        w, n = surf_rec[(p, s)]
        return (w + 1) / (n + 2)
    def fatigue(p):
        return sum(1 for x in dates[p] if 0 < (d - x).days <= 14)

    rows.append({
        "elo_diff": elo[a] - elo[b],
        "elo_surf_diff": elo_surf[(a, s)] - elo_surf[(b, s)],
        "form_diff": form(a) - form(b),
        "surf_wr_diff": surf_wr(a) - surf_wr(b),
        "fatigue_diff": fatigue(a) - fatigue(b),
        "exp_diff": np.log1p(len(dates[a])) - np.log1p(len(dates[b])),
    })

    # 2) UPDATE: learn from the result
    ea = expected(elo[a], elo[b])
    elo[a] += K * (y - ea)
    elo[b] -= K * (y - ea)

    es = expected(elo_surf[(a, s)], elo_surf[(b, s)])
    elo_surf[(a, s)] += K * (y - es)
    elo_surf[(b, s)] -= K * (y - es)

    recent[a].append(y); recent[b].append(1 - y)
    surf_rec[(a, s)][0] += y;     surf_rec[(a, s)][1] += 1
    surf_rec[(b, s)][0] += 1 - y; surf_rec[(b, s)][1] += 1
    dates[a].append(d); dates[b].append(d)

feats = pd.DataFrame(rows, index=df.index)

# Ranking: log because rank 1 vs 10 matters far more than 101 vs 110
df["A_rank"] = df["A_rank"].fillna(2000)
df["B_rank"] = df["B_rank"].fillna(2000)
feats["log_rank_diff"] = np.log(df["A_rank"]) - np.log(df["B_rank"])

df = pd.concat([df, feats], axis=1)
FEATURES = list(feats.columns)

# Odds are only needed for evaluation, so filter AFTER building player histories
n_before = len(df)
df = df[(df["A_odds"] > 1) & (df["B_odds"] > 1)].copy()
print("Dropped for invalid/missing odds:", n_before - len(df))

print(df[FEATURES].describe().T[["mean", "std", "min", "max"]])
print(df[FEATURES + ["y"]].corr()["y"].sort_values())

Dropped for invalid/missing odds: 93
                   mean         std         min         max
elo_diff       0.778471  158.757527 -737.330033  742.747303
elo_surf_diff  1.087717  132.611106 -662.941434  742.055824
form_diff      0.000626    0.239725   -1.000000    1.000000
surf_wr_diff   0.000855    0.163390   -0.732143    0.623529
fatigue_diff   0.003012    1.999932  -10.000000   10.000000
exp_diff      -0.018200    1.478863   -5.680173    5.828946
log_rank_diff  0.002301    1.482231   -7.113753    6.295266
log_rank_diff   -0.341464
fatigue_diff     0.107827
exp_diff         0.210984
form_diff        0.224551
surf_wr_diff     0.255092
elo_surf_diff    0.318402
elo_diff         0.341713
y                1.000000
Name: y, dtype: float64


Elo and rank are the strongest signals and largely overlap; fatigue_diff has the “wrong” sign because it captures momentum (deep tournament runs), not tiredness.

## 4. Time-based split, scaling and baselines
2018 = warm-up only, 2019-2023 = train, 2024 = validation, 2025 = test.

In [21]:
train = df[df["season"].between(2019, 2023)]
val   = df[df["season"] == 2024]
test  = df[df["season"] == 2025]
print(len(train), len(val), len(test))

# Fit the scaler on TRAIN ONLY, then apply it everywhere
scaler = StandardScaler().fit(train[FEATURES])
X_train = scaler.transform(train[FEATURES])
X_val   = scaler.transform(val[FEATURES])
X_test  = scaler.transform(test[FEATURES])
y_train, y_val, y_test = train["y"].values, val["y"].values, test["y"].values

def bookmaker_prob(d):
    inv_a, inv_b = 1 / d["A_odds"], 1 / d["B_odds"]
    return inv_a / (inv_a + inv_b)          # remove the bookmaker's margin

rank_pred = (test["A_rank"] < test["B_rank"]).astype(int)
book_p = bookmaker_prob(test)

print("Higher-ranked player accuracy:", accuracy_score(y_test, rank_pred))
print("Bookmaker favourite accuracy: ", accuracy_score(y_test, (book_p > 0.5).astype(int)))
print("Bookmaker log loss:           ", log_loss(y_test, book_p))

11276 2601 2504
Higher-ranked player accuracy: 0.6477635782747604
Bookmaker favourite accuracy:  0.6757188498402555
Bookmaker log loss:            0.5929440674190593


Ranking gets 64.8%, the bookmaker 67.6% with log loss 0.593. That's the bar.

## 5. Logistic regression three ways
Same model, same split, same regularisation (sklearn's C translated to λ = 1/(2·C·n) for PyTorch).

In [22]:
from sklearn.linear_model import LogisticRegression

for C in [0.001, 0.01, 0.1, 1, 10]:
    m = LogisticRegression(C=C).fit(X_train, y_train)
    p = m.predict_proba(X_val)[:, 1]
    print(f"C={C:<6} val log loss={log_loss(y_val, p):.4f}  val acc={accuracy_score(y_val, p > 0.5):.4f}")

C=0.001  val log loss=0.6125  val acc=0.6551
C=0.01   val log loss=0.6111  val acc=0.6555
C=0.1    val log loss=0.6111  val acc=0.6567
C=1      val log loss=0.6111  val acc=0.6571
C=10     val log loss=0.6111  val acc=0.6571


**Choice of C.** Validation log loss stays flat between C = 0.01 and C = 10 (0.6111), and only C = 0.001 performs worse (0.6125, a sign of underfitting). With roughly 11k training rows and just 7 features the model has little room to overfit, so at this data size the regularisation strength makes almost no difference. I selected **C = 0.01**, the strongest regularisation that leaves validation log loss unchanged, which gives the most conservative weights at no cost in performance. I tune on log loss instead of accuracy because the betting decision relies on well-calibrated probabilities, not only on identifying the favourite.

In [24]:
BEST_C = 0.01

sk_model = LogisticRegression(C=BEST_C).fit(X_train, y_train)

# Translate sklearn's C into the equivalent penalty for a mean-loss PyTorch objective
lam = 1 / (2 * BEST_C * len(X_train))
print("lambda for PyTorch:", lam)

lambda for PyTorch: 0.004434196523589925


In [26]:
import torch
import torch.nn as nn

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.float32)
Xva = torch.tensor(X_val, dtype=torch.float32)
n_features = Xtr.shape[1]

def train_manual(lr, epochs=2000):
    w = torch.zeros(n_features, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    for epoch in range(epochs):
        p = torch.sigmoid(Xtr @ w + b)
        p = p.clamp(1e-7, 1 - 1e-7)                     # avoid log(0)
        bce = -(ytr * torch.log(p) + (1 - ytr) * torch.log(1 - p)).mean()
        loss = bce + lam * (w ** 2).sum()               # bias not penalised, like sklearn
        loss.backward()                                 # autograd computes the gradients
        with torch.no_grad():                           # manual gradient step
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    return w.detach(), b.detach()

for lr in [0.01, 0.1, 1.0]:
    w, b = train_manual(lr)
    p_val = torch.sigmoid(Xva @ w + b).numpy()
    print(f"lr={lr:<5} val log loss={log_loss(y_val, p_val):.4f}")

lr=0.01  val log loss=0.6113
lr=0.1   val log loss=0.6111
lr=1.0   val log loss=0.6111


**Learning rate (manual loop).** With lr = 0.1 and lr = 1.0 the manual loop reaches the same validation log loss as sklearn (0.6111), while lr = 0.01 falls slightly short (0.6113) because 2,000 small steps are not enough for full convergence. I selected **lr = 1.0**, the largest stable step size and therefore the one most likely to converge fully within the fixed iteration budget. A step this large is only stable because the features were standardised beforehand.

In [27]:
LR_MANUAL = 1.0   # chosen on validation log loss (see above)
w_manual, b_manual = train_manual(LR_MANUAL)

In [28]:
class LogReg(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.linear = nn.Linear(n, 1)                   # holds the weights and bias
    def forward(self, x):
        return self.linear(x).squeeze(1)                # raw scores (logits)

def train_module(lr, epochs=2000):
    torch.manual_seed(42)                               # nn.Linear starts from random weights
    model = LogReg(n_features)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()                    # sigmoid + log loss in one, numerically stable
    for epoch in range(epochs):
        opt.zero_grad()
        loss = loss_fn(model(Xtr), ytr) + lam * (model.linear.weight ** 2).sum()  # bias not penalised
        loss.backward()
        opt.step()
    return model

for lr in [0.01, 0.1, 1.0]:
    model = train_module(lr)
    with torch.no_grad():
        p_val = torch.sigmoid(model(Xva)).numpy()
    print(f"lr={lr:<5} val log loss={log_loss(y_val, p_val):.4f}")

lr=0.01  val log loss=0.6111
lr=0.1   val log loss=0.6111
lr=1.0   val log loss=0.6111


**Learning rate (nn.Module).** All three learning rates reach a validation log loss of 0.6111. Unlike in the manual loop, even lr = 0.01 converges here, most likely because nn.Linear initialises the weights with small random values instead of zeros, and this starting point happened to lie closer to the optimum. I selected **lr = 1.0**, the same value as in the manual loop, so that the two PyTorch versions differ only in their implementation and not in their settings.

In [29]:
LR_MODULE = 1.0   # same as manual loop, for a like-for-like comparison
nn_model = train_module(LR_MODULE)

## 6. Three-method comparison on the 2025 test season

In [30]:
# 1) Learned weights side by side
coefs = pd.DataFrame({
    "sklearn": sk_model.coef_[0],
    "manual_torch": w_manual.numpy(),
    "nn_module": nn_model.linear.weight.detach().numpy()[0],
}, index=FEATURES)
coefs.loc["intercept"] = [sk_model.intercept_[0], b_manual.item(), nn_model.linear.bias.item()]
print(coefs.round(4))

# 2) Test-set predictions from each method
Xte = torch.tensor(X_test, dtype=torch.float32)
p_sk = sk_model.predict_proba(X_test)[:, 1]
with torch.no_grad():
    p_manual = torch.sigmoid(Xte @ w_manual + b_manual).numpy()
    p_nn = torch.sigmoid(nn_model(Xte)).numpy()

print("\nMax difference in predicted probability vs sklearn:")
print("  manual torch:", np.abs(p_sk - p_manual).max())
print("  nn.Module:   ", np.abs(p_sk - p_nn).max())

# 3) Results table: three methods vs baselines, same test season
def row(name, p=None, pred=None):
    pred = (p > 0.5).astype(int) if pred is None else pred
    return {"model": name,
            "accuracy": accuracy_score(y_test, pred),
            "log_loss": log_loss(y_test, p) if p is not None else np.nan}

results = pd.DataFrame([
    row("Naive: higher-ranked player", pred=rank_pred),
    row("Bookmaker (Bet365, margin removed)", p=book_p.values),
    row("Logistic regression: sklearn", p=p_sk),
    row("Logistic regression: manual PyTorch", p=p_manual),
    row("Logistic regression: nn.Module", p=p_nn),
])
print()
print(results.round(4).to_string(index=False))

               sklearn  manual_torch  nn_module
elo_diff        0.2463        0.2463     0.2463
elo_surf_diff   0.2438        0.2435     0.2435
form_diff       0.0203        0.0205     0.0205
surf_wr_diff    0.0563        0.0563     0.0563
fatigue_diff    0.1138        0.1138     0.1138
exp_diff        0.0317        0.0317     0.0317
log_rank_diff  -0.3594       -0.3594    -0.3594
intercept      -0.0061       -0.0060    -0.0060

Max difference in predicted probability vs sklearn:
  manual torch: 0.00019870770179009067
  nn.Module:    0.00019870770179009067

                              model  accuracy  log_loss
        Naive: higher-ranked player    0.6478       NaN
 Bookmaker (Bet365, margin removed)    0.6757    0.5929
       Logistic regression: sklearn    0.6534    0.6176
Logistic regression: manual PyTorch    0.6534    0.6176
     Logistic regression: nn.Module    0.6534    0.6176


**Comparison.** The three implementations produce the same results on the 2025 test season: identical accuracy (65.34%) and log loss (0.6176), with a maximum difference in predicted probabilities of 0.0002. The only small differences in coefficients appear on elo_surf_diff and form_diff, both of which are correlated with elo_diff. Gradient descent with a fixed number of steps converges slowly along correlated directions, whereas sklearn's L-BFGS solver converges to the optimum within its tolerance. The model outperforms the naive rule of always backing the higher-ranked player (64.8%) but not the bookmaker (67.6%, log loss 0.593), so overall it does not predict outcomes better than the market. Whether it can still identify profitable bets is tested in the next section.

## 7. Business evaluation: value-betting profit
Each match gives two candidate bets (back A or back B). A bet is placed when expected value = model probability × odds − 1 exceeds a minimum edge threshold.
Flat stake of 1 unit per bet. The threshold is chosen on 2024 (validation) and then applied once to 2025 (test).

In [35]:
def candidate_bets(d, p):
    """Turn each match into two candidate bets: back A, back B."""
    a = pd.DataFrame({"prob": p,     "odds": d["A_odds"].values, "won": d["y"].values})
    b = pd.DataFrame({"prob": 1 - p, "odds": d["B_odds"].values, "won": 1 - d["y"].values})
    c = pd.concat([a, b], ignore_index=True)
    c["ev"] = c["prob"] * c["odds"] - 1                      # model's expected value per unit staked
    c["pnl"] = np.where(c["won"] == 1, c["odds"] - 1, -1.0)  # what the bet would actually return
    return c

def betting_summary(c, threshold):
    placed = c["ev"] > threshold
    n = placed.sum()
    profit = c.loc[placed, "pnl"].sum()
    return {"threshold": threshold, "bets": n, "profit": round(profit, 2),
            "roi": round(profit / n, 4) if n else np.nan,
            "win_rate": round(c.loc[placed, "won"].mean(), 4) if n else np.nan,
            "avg_odds": round(c.loc[placed, "odds"].mean(), 2) if n else np.nan}  # >2 = mostly underdogs

# Choose the threshold on VALIDATION (2024)
p_val_sk = sk_model.predict_proba(X_val)[:, 1]
val_bets = candidate_bets(val, p_val_sk)
print("Validation 2024:")
print(pd.DataFrame([betting_summary(val_bets, t) for t in [0, 0.02, 0.05, 0.1, 0.15, 0.2]]).to_string(index=False))

# Reference: blindly backing every bookmaker favourite, per season
def favourite_roi(d):
    fav_is_a = d["A_odds"].values < d["B_odds"].values
    fav_odds = np.where(fav_is_a, d["A_odds"], d["B_odds"])
    fav_won = np.where(fav_is_a, d["y"] == 1, d["y"] == 0)
    return round(np.where(fav_won, fav_odds - 1, -1.0).mean(), 4)

print("\nReference, back every favourite: 2024 ROI =", favourite_roi(val), "| 2025 ROI =", favourite_roi(test))

Validation 2024:
 threshold  bets  profit     roi  win_rate  avg_odds
      0.00  2015 -289.50 -0.1437    0.3831      3.31
      0.02  1787 -264.44 -0.1480    0.3654      3.47
      0.05  1494 -227.50 -0.1523    0.3400      3.71
      0.10  1152 -218.29 -0.1895    0.2986      4.06
      0.15   908 -177.94 -0.1960    0.2687      4.48
      0.20   705 -150.22 -0.2131    0.2411      4.93

Reference, back every favourite: 2024 ROI = -0.0339 | 2025 ROI = -0.052


**Validation result.** Every edge threshold loses money on the 2024 season (ROI between −14% and −21%), which is worse than simply backing the bookmaker's favourite in every match of the same season (−3.4%). ROI actually *deteriorates* as the threshold increases: the matches where the model disagrees most with the bookmaker are mostly errors of the model, because the market prices in information the model does not have (injuries, withdrawals, playing conditions). These bets are concentrated on underdogs: the average odds rise from 3.31 at threshold 0 to 4.93 at threshold 0.20, while the win rate drops from 38% to 24%. Longshots are where bookmakers typically embed the largest margin (the "favourite-longshot bias"), so a strategy that drifts towards them is penalised accordingly. Following the rule "select on validation, apply once to test", I set the threshold to 0 (the best validation ROI) and evaluate it a single time on 2025.

In [32]:
EDGE_THRESHOLD = 0.0   # chosen on validation (best 2024 ROI)

test_bets = candidate_bets(test, p_sk)
print("Test 2025:", betting_summary(test_bets, EDGE_THRESHOLD))

Test 2025: {'threshold': 0.0, 'bets': np.int64(1926), 'profit': np.float64(-207.19), 'roi': np.float64(-0.1076), 'win_rate': np.float64(0.3941)}


**Test result (2025, evaluated once).** At threshold 0 the strategy places 1,926 bets, wins 39.4% of them and loses 207 units, for an **ROI of −10.8%**. Backing every bookmaker favourite in the same season loses only 5.2%, roughly equal to the bookmaker's margin, so the model's selections cost about twice as much as using no model at all. The test ROI is slightly less negative than the validation ROI (−14.4%), but both seasons point in the same direction, and a single season of bets is too small a sample to draw conclusions from that gap. **Business conclusion:** a model based only on public ranking and form data should not be used for value betting on the ATP main tour. The market already prices in this information, and the cases where the model disagrees most strongly with the market are mostly its own errors.

## 8. Save artifacts for the dashboard

In [33]:
import os, json, joblib

os.makedirs("models", exist_ok=True)

joblib.dump(sk_model, "models/sklearn_logreg.joblib")
joblib.dump(scaler, "models/scaler.joblib")
torch.save({"w": w_manual, "b": b_manual}, "models/manual_torch.pt")
torch.save(nn_model.state_dict(), "models/nn_module.pt")

# Data the dashboard displays: 2019-2025 rows with raw features, odds and outcomes
cols = ["Date", "Tournament", "Surface", "Round", "A", "B", "A_rank", "B_rank",
        "A_odds", "B_odds", "y", "season"] + FEATURES
df[df["season"] >= 2019][cols].to_csv("models/eval_data.csv", index=False)

config = {"features": FEATURES, "best_c": BEST_C, "lambda": lam,
          "lr_manual": LR_MANUAL, "lr_module": LR_MODULE,
          "edge_threshold": EDGE_THRESHOLD}
with open("models/config.json", "w") as f:
    json.dump(config, f, indent=2)

print(os.listdir("models"))

['sklearn_logreg.joblib', 'eval_data.csv', 'nn_module.pt', 'scaler.joblib', 'config.json', 'manual_torch.pt']
